In [1]:
import torch

In [2]:
from Circuits import Circuits
circuits=Circuits()
graph_data,text_data= circuits.data_lodder()


Loading dataset files...
Loaded dataset files successfully.


In [3]:
import GtoTmodel
device="cuda"
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 10 # Number of transformer layers
dropout = 0.1  # Dropout rate
graph_input_dim = 310  # Number of colomns in the graph
text_vocab_size = 894  # Vocabulary size for text

model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

model = model.to(device)

c:\Users\MSI\miniconda3\envs\ml\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [4]:
graph_dataset = torch.cat(graph_data).to(device)
text_dataset = torch.cat(text_data,).to(device)
graph_dataset.shape,text_dataset.shape

(torch.Size([351461, 310]), torch.Size([351461]))

In [30]:
import torch
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm

# Hyperparameters
learning_rate = 0.001
num_epochs = 50
batch_size = 32

# Loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Assuming 0 is the padding index
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Prepare data
graph_batches = [graph_data[i:i + batch_size] for i in range(0, len(graph_data), batch_size)]
text_batches = [text_data[i:i + batch_size] for i in range(0, len(text_data), batch_size)]

print(f"Number of batches: {len(graph_batches)}")

# Training loop
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for graph_batch, text_batch in tqdm(zip(graph_batches, text_batches), total=len(graph_batches)):
        # Move data to device
        
        text_batch = torch.nn.utils.rnn.pad_sequence(text_batch, batch_first=True).to(device)
        graph_batch = torch.nn.utils.rnn.pad_sequence(graph_batch, batch_first=True).to(device)

        # Prepare input and target for the decoder
        decoder_input = text_batch[:, :-1]  # All except the last token
        decoder_target = text_batch[:, 1:]  # All except the first token

        # Forward pass
        outputs = model(graph_batch, decoder_input)

        # Compute loss
        outputs = outputs.reshape(-1, outputs.size(-1))  # Flatten for CrossEntropyLoss
        decoder_target = decoder_target.reshape(-1).long()  # Flatten target and cast to Long
        loss = criterion(outputs, decoder_target)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss / len(graph_batches):.4f}")

print("Training complete.")

Number of batches: 105


100%|██████████| 105/105 [00:29<00:00,  3.57it/s]


Epoch 1/50, Loss: 0.1506


100%|██████████| 105/105 [00:28<00:00,  3.62it/s]


Epoch 2/50, Loss: 0.1485


...

Epoch 30/50, Loss: 0.1389


100%|██████████| 105/105 [00:28<00:00,  3.63it/s]


Epoch 31/50, Loss: 0.1387


  9%|▊         | 9/105 [00:02<00:29,  3.30it/s]


KeyboardInterrupt: 

In [42]:
# Define the file path to save the model and hyperparameters
save_path = "model_checkpoint.pth"

# Create a dictionary to store the model state and hyperparameters
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'embed_dim': embed_dim,
    'num_heads': num_heads,
    'num_layers': num_layers,
    'dropout': dropout,
    'learning_rate': learning_rate,
    'text_vocab_size': text_vocab_size,
    'graph_input_dim': graph_input_dim,
}

# Save the checkpoint
torch.save(checkpoint, save_path)
print(f"Model and hyperparameters saved to {save_path}")

Model and hyperparameters saved to model_checkpoint.pth


In [26]:
graph_data[1].shape

torch.Size([127, 310])

In [41]:
# Load the saved model checkpoint
checkpoint_path = "model_checkpoint.pth"
checkpoint = torch.load(checkpoint_path)

# Reinitialize the model with the saved hyperparameters
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim=checkpoint['graph_input_dim'],
    text_vocab_size=checkpoint['text_vocab_size'],
    embed_dim=checkpoint['embed_dim'],
    num_heads=checkpoint['num_heads'],
    num_layers=checkpoint['num_layers'],
    dropout=checkpoint['dropout']
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Select a sample graph input
sample_graph = graph_data[0].unsqueeze(0).to(device)  # Add batch dimension

# Generate a sequence
start_token = torch.tensor([82], dtype=torch.long).to(device)  # Assuming 0 is the start token
generated_sequence = [start_token.item()]

for _ in range(5000):  # Generate up to 50 tokens
    input_sequence = torch.tensor(generated_sequence, dtype=torch.long).unsqueeze(0).to(device)
    output = model(sample_graph, input_sequence)
    next_token = torch.argmax(output[:, -1, :], dim=-1).item()  # Get the most probable next token
    generated_sequence.append(next_token)
    if next_token == 892:  # Assuming 892 is the end token
        break

# Convert indices back to components
generated_text = circuits.get_component_fromlist(generated_sequence)

print("Generated Sequence:", generated_text)

C:\Users\MSI\AppData\Local\Temp\ipykernel_13192\3126809034.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Generated Sequence: ['PNP11', 'VRF2', 'VLO1', 'VLO2', 'NPN10', 'NPN10_C', 'NPN10_B', 'NM3', 'NM3_D', 'NM3_G', 'NM3_S', 'NM3_B', 'NM4', 'NM4_D', 'NM4_G', 'NM4_S', 'NM4_B', 'NM5', 'NM5_D', 'NM5_G', 'NM5_S', 'NM5_B', 'NM6', 'NM6_D', 'NM6_G', 'NM6_S', 'NM6_B', 'NM7', 'NM7_D', 'NM7_G', 'NM7_S', 'NM7_B', 'NM8', 'NM8_D', 'NM8_G', 'NM8_S', 'NM8_B', 'NM9', 'NM9_D', 'NM9_G', 'NM9_S', 'NM9_B', 'NM10', 'NM10_D', 'NM10_G', 'NM10_S', 'NM10_B', 'NM11', 'NM11_D', 'NM11_G', 'NM11_S', 'NM11_B', 'NM12', 'NM12_D', 'NM12_G', 'NM12_S', 'NM12_B', 'NM13', 'NM13_D', 'NM13_G', 'NM13_S', 'NM13_B', 'NM14', 'NM14_D', 'NM14_G', 'NM14_S', 'NM14_B', 'NM15', 'NM15_D', 'NM15_G', 'NM15_S', 'NM15_B', 'NM16', 'NM16_D', 'NM16_G', 'NM16_S', 'NM16_B', 'end']
